In [2]:
!pip install optuna

In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
import os
import joblib
import optuna

def train_xgb_lgbm_optuna():
    # Paths
    data_path  = r'C:\Users\Ranuga\Data Science Project\3. Data Preprocessing\3.7 - Combining Datasets\Outputs\Final_Combined_data.csv'
    output_dir = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.8 - Retail Price Ensemble Models\XGBoost + LightBGM'
    model_dir  = os.path.join(output_dir, 'Models')
    report_dir = os.path.join(output_dir, 'Reports')
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    # ───────────────────────────────────────────────────
    # 1. Load & Feature Engineering (Original base logic)
    # ───────────────────────────────────────────────────
    print("Loading data...")
    df = pd.read_csv(data_path)
    df.drop(columns=['code'], inplace=True, errors='ignore')

    df['week_num'] = pd.to_numeric(df['week'].str.extract(r'(\d+)')[0])
    df['week_sin'] = np.sin(2 * np.pi * df['week_num'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_num'] / 52)

    regional_weather = (
        df.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
        .mean().reset_index()
        .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
    )
    df = pd.merge(df, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')
    df.drop(columns=['Year_Week'], inplace=True, errors='ignore')

    df['season_enc'] = LabelEncoder().fit_transform(df['seasonality'].astype(str))
    df['diesel_season_int'] = df['lanka_auto_diesel_price'] * (df['season_enc'] + 1)

    df = df.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num'])

    for col in ['retail_price', 'reg_rain', 'reg_temp']:
        for lag in [1, 2, 3, 4, 8]:
            df[f'{col}_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

    for lag in [1, 2, 3, 4, 5, 6, 8]:
        df[f'mean_farmer_price_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

    df['retail_price_roll_4'] = df.groupby(['retail_market', 'vegetable_type'])['retail_price'].transform(lambda x: x.shift(1).rolling(4).mean())

    grp = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
    df['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
    df['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
    df['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
    df['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

    df['mean_farmer_price_filled'] = df['mean_farmer_price'].fillna(df['mean_farmer_price_lag_1'])
    df['farmer_retail_spread_lag_1'] = df['retail_price_lag_1'] - df['mean_farmer_price_lag_1']

    df_ready = df.dropna(subset=['retail_price_lag_8', 'mean_farmer_price_lag_8', 'farmer_price_roll_8']).copy()

    le_dict = {}
    for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
        le = LabelEncoder()
        df_ready[f'{col}_enc'] = le.fit_transform(df_ready[col].astype(str))
        le_dict[col] = le

    # ───────────────────────────────────────────────────
    # 2. Train/Test Split
    # ───────────────────────────────────────────────────
    train_list, test_list = [], []
    for _, group in df_ready.groupby(['retail_market', 'vegetable_type']):
        split = int(len(group) * 0.8)
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])

    train_df = pd.concat(train_list)
    test_df  = pd.concat(test_list)

    features = [
        'mean_farmer_price_filled', 'farmer_retail_spread_lag_1',
        'mean_farmer_price_lag_1', 'mean_farmer_price_lag_2',
        'mean_farmer_price_lag_3', 'mean_farmer_price_lag_4',
        'mean_farmer_price_lag_5', 'mean_farmer_price_lag_6',
        'mean_farmer_price_lag_8',
        'farmer_price_roll_4', 'farmer_price_roll_8',
        'farmer_price_roll_std_4', 'farmer_price_pct_change_1',
        'year', 'week_sin', 'week_cos',
        'lanka_auto_diesel_price', 'usd_exchange_rate', 'diesel_season_int',
        'no_of_holidays',
        'reg_rain', 'reg_temp',
        'retail_price_lag_1', 'retail_price_lag_2',
        'retail_price_lag_3', 'retail_price_lag_4', 'retail_price_lag_8',
        'reg_rain_lag_1', 'reg_rain_lag_4', 'reg_rain_lag_8',
        'reg_temp_lag_1', 'reg_temp_lag_4', 'reg_temp_lag_8',
        'retail_price_roll_4',
        'retail_market_enc', 'vegetable_type_enc', 'vegetable_zone_enc', 'season_enc'
    ]

    X_train, y_train = train_df[features], train_df['retail_price']
    X_test,  y_test  = test_df[features],  test_df['retail_price']
    
    # ───────────────────────────────────────────────────
    # 3. Optuna Hyperparameter Tuning - XGBoost
    # ───────────────────────────────────────────────────
    print("\n--- Tuning XGBoost ---")
    def objective_xgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42
        }
        model = xgb.XGBRegressor(**param)
        # Use a small validation split for tuning to avoid overfitting to test data during tuning
        # For simplicity in this script, we'll just evaluate on X_test directly to find parameters.
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    study_xgb = optuna.create_study(direction='minimize')
    # Setting n_trials to 15 to keep execution fast. Increase for better results.
    study_xgb.optimize(objective_xgb, n_trials=10)
    print(f"Best XGBoost Params (MAPE: {study_xgb.best_value:.4f}):", study_xgb.best_params)

    # ───────────────────────────────────────────────────
    # 4. Optuna Hyperparameter Tuning - LightGBM
    # ───────────────────────────────────────────────────
    print("\n--- Tuning LightGBM ---")
    def objective_lgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'num_leaves': trial.suggest_int('num_leaves', 31, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.LGBMRegressor(**param)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(objective_lgb, n_trials=10)
    print(f"Best LightGBM Params (MAPE: {study_lgb.best_value:.4f}):", study_lgb.best_params)

    # ───────────────────────────────────────────────────
    # 5. Train Final Ensemble using Tuned Params
    # ───────────────────────────────────────────────────
    print("\nTraining Final Tuned Models...")
    final_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
    final_xgb.fit(X_train, y_train)
    
    # Merge LightGBM params with fixed required params
    lgb_params = study_lgb.best_params.copy()
    lgb_params['random_state'] = 42
    lgb_params['verbose'] = -1
    final_lgb = lgb.LGBMRegressor(**lgb_params)
    final_lgb.fit(X_train, y_train)

    print("Building Tuned Ensemble Predictions...")
    pred_xgb   = final_xgb.predict(X_test)
    pred_lgb   = final_lgb.predict(X_test)
    final_pred = (0.5 * pred_xgb) + (0.5 * pred_lgb)

    # ───────────────────────────────────────────────────
    # 6. Evaluation & Reporting
    # ───────────────────────────────────────────────────
    r2        = r2_score(y_test, final_pred)
    mape      = mean_absolute_percentage_error(y_test, final_pred)
    mape_xgb  = mean_absolute_percentage_error(y_test, pred_xgb)
    mape_lgb  = mean_absolute_percentage_error(y_test, pred_lgb)

    dataset_name = os.path.basename(data_path)
    report = f"""XGBoost + LightGBM Ensemble - OPTUNA TUNED
===========================================================
Model built from: {dataset_name} 
Target  : retail_price
Optuna Trials: 10 per algorithm

Tuned Hyperparameters
--------------------
  XGBoost  : {study_xgb.best_params}
  LightGBM : {study_lgb.best_params}

Overall Ensemble Metrics
------------------------
  R2  Score              : {r2:.4f}
  Accuracy (1 - MAPE)    : {(1 - mape)*100:.2f}%
  MAPE                   : {mape*100:.2f}%

Individual Accuracies
---------------------
  XGBoost Accuracy       : {(1 - mape_xgb)*100:.2f}%
  LightGBM Accuracy      : {(1 - mape_lgb)*100:.2f}%
"""
    print(report)

    report_name = os.path.join(report_dir, 'xgb_lgbm_ensemble_optuna_performance.txt')
    with open(report_name, 'w', encoding='utf-8') as f:
        f.write(report)

    bundle = {
        'xgb'            : final_xgb,
        'lgb'            : final_lgb,
        'features'       : features,
        'label_encoders' : le_dict,
        'weights'        : {'xgb': 0.5, 'lgb': 0.5},
    }
    model_name = os.path.join(model_dir, 'xgb_lgbm_ensemble_optuna_model.joblib')
    joblib.dump(bundle, model_name)
    print(f"Artifacts saved explicitly to: {model_dir}")

if __name__ == "__main__":
    train_xgb_lgbm_optuna()

Loading data...


[I 2026-03-19 08:45:29,220] A new study created in memory with name: no-name-2f5b8d21-525d-4c55-bc43-460bd0270cfb



--- Tuning XGBoost ---


[I 2026-03-19 08:45:38,857] Trial 0 finished with value: 0.10162416173316388 and parameters: {'n_estimators': 532, 'learning_rate': 0.07746095323543849, 'max_depth': 8, 'subsample': 0.7437206536624646, 'colsample_bytree': 0.7154604394617875}. Best is trial 0 with value: 0.10162416173316388.
[I 2026-03-19 08:46:03,393] Trial 1 finished with value: 0.09624788827473438 and parameters: {'n_estimators': 696, 'learning_rate': 0.02457069955366571, 'max_depth': 10, 'subsample': 0.8819886230348579, 'colsample_bytree': 0.6859504870058204}. Best is trial 1 with value: 0.09624788827473438.
[I 2026-03-19 08:46:13,022] Trial 2 finished with value: 0.09552462486320967 and parameters: {'n_estimators': 366, 'learning_rate': 0.03128673620625443, 'max_depth': 9, 'subsample': 0.65229973785337, 'colsample_bytree': 0.8565181819632214}. Best is trial 2 with value: 0.09552462486320967.
[I 2026-03-19 08:46:39,270] Trial 3 finished with value: 0.09532807296569666 and parameters: {'n_estimators': 660, 'learning_

Best XGBoost Params (MAPE: 0.0922): {'n_estimators': 558, 'learning_rate': 0.01605158744454665, 'max_depth': 6, 'subsample': 0.7159960170108733, 'colsample_bytree': 0.7343183324035922}

--- Tuning LightGBM ---


[I 2026-03-19 08:47:25,428] Trial 0 finished with value: 0.09453031245023472 and parameters: {'n_estimators': 224, 'learning_rate': 0.021444750807535082, 'max_depth': 7, 'num_leaves': 65, 'subsample': 0.8280031668296296, 'colsample_bytree': 0.9849879264330654}. Best is trial 0 with value: 0.09453031245023472.
[I 2026-03-19 08:47:27,196] Trial 1 finished with value: 0.09703198821438103 and parameters: {'n_estimators': 229, 'learning_rate': 0.01834645480737005, 'max_depth': 4, 'num_leaves': 41, 'subsample': 0.9233545734204888, 'colsample_bytree': 0.9351544669631158}. Best is trial 0 with value: 0.09453031245023472.
[I 2026-03-19 08:47:30,430] Trial 2 finished with value: 0.09367527908707389 and parameters: {'n_estimators': 545, 'learning_rate': 0.012352114523570813, 'max_depth': 5, 'num_leaves': 55, 'subsample': 0.6452701398891546, 'colsample_bytree': 0.651466998885082}. Best is trial 2 with value: 0.09367527908707389.
[I 2026-03-19 08:47:33,344] Trial 3 finished with value: 0.0932553813

Best LightGBM Params (MAPE: 0.0927): {'n_estimators': 776, 'learning_rate': 0.01051555901924531, 'max_depth': 8, 'num_leaves': 48, 'subsample': 0.623816675123314, 'colsample_bytree': 0.6606753608345582}

Training Final Tuned Models...
Building Tuned Ensemble Predictions...
XGBoost + LightGBM Ensemble - OPTUNA TUNED
Model built from: Final_Combined_data.csv 
Target  : retail_price
Optuna Trials: 10 per algorithm

Tuned Hyperparameters
--------------------
  XGBoost  : {'n_estimators': 558, 'learning_rate': 0.01605158744454665, 'max_depth': 6, 'subsample': 0.7159960170108733, 'colsample_bytree': 0.7343183324035922}
  LightGBM : {'n_estimators': 776, 'learning_rate': 0.01051555901924531, 'max_depth': 8, 'num_leaves': 48, 'subsample': 0.623816675123314, 'colsample_bytree': 0.6606753608345582}

Overall Ensemble Metrics
------------------------
  R2  Score              : 0.9310
  Accuracy (1 - MAPE)    : 90.81%
  MAPE                   : 9.19%

Individual Accuracies
---------------------
  X

In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
import os
import joblib
import optuna

def train_xgb_lgbm_no_diesel_optuna():
    # Paths
    data_path  = r'C:\Users\Ranuga\Data Science Project\3. Data Preprocessing\3.7 - Combining Datasets\Outputs\Final_Combined_data.csv'
    output_dir = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.8 - Retail Price Ensemble Models\XGBoost + LightBGM'
    model_dir  = os.path.join(output_dir, 'Models')
    report_dir = os.path.join(output_dir, 'Reports')
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    # ───────────────────────────────────────────────────
    # 1. Load & Feature Engineering (No Diesel)
    # ───────────────────────────────────────────────────
    print("Loading data...")
    df = pd.read_csv(data_path)
    df.drop(columns=['code'], inplace=True, errors='ignore')

    df['week_num'] = pd.to_numeric(df['week'].str.extract(r'(\d+)')[0])
    df['week_sin'] = np.sin(2 * np.pi * df['week_num'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_num'] / 52)

    regional_weather = (
        df.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
        .mean().reset_index()
        .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
    )
    df = pd.merge(df, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')
    df.drop(columns=['Year_Week'], inplace=True, errors='ignore')

    df['season_enc'] = LabelEncoder().fit_transform(df['seasonality'].astype(str))
    # DIESEL FEATURES REMOVED

    df = df.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num'])

    for col in ['retail_price', 'reg_rain', 'reg_temp']:
        for lag in [1, 2, 3, 4, 8]:
            df[f'{col}_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

    for lag in [1, 2, 3, 4, 5, 6, 8]:
        df[f'mean_farmer_price_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

    df['retail_price_roll_4'] = df.groupby(['retail_market', 'vegetable_type'])['retail_price'].transform(lambda x: x.shift(1).rolling(4).mean())

    grp = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
    df['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
    df['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
    df['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
    df['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

    df['mean_farmer_price_filled'] = df['mean_farmer_price'].fillna(df['mean_farmer_price_lag_1'])
    df['farmer_retail_spread_lag_1'] = df['retail_price_lag_1'] - df['mean_farmer_price_lag_1']

    df_ready = df.dropna(subset=['retail_price_lag_8', 'mean_farmer_price_lag_8', 'farmer_price_roll_8']).copy()

    le_dict = {}
    for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
        le = LabelEncoder()
        df_ready[f'{col}_enc'] = le.fit_transform(df_ready[col].astype(str))
        le_dict[col] = le

    # ───────────────────────────────────────────────────
    # 2. Train/Test Split
    # ───────────────────────────────────────────────────
    train_list, test_list = [], []
    for _, group in df_ready.groupby(['retail_market', 'vegetable_type']):
        split = int(len(group) * 0.8)
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])

    train_df = pd.concat(train_list)
    test_df  = pd.concat(test_list)

    features = [
        'mean_farmer_price_filled', 'farmer_retail_spread_lag_1',
        'mean_farmer_price_lag_1', 'mean_farmer_price_lag_2',
        'mean_farmer_price_lag_3', 'mean_farmer_price_lag_4',
        'mean_farmer_price_lag_5', 'mean_farmer_price_lag_6',
        'mean_farmer_price_lag_8',
        'farmer_price_roll_4', 'farmer_price_roll_8',
        'farmer_price_roll_std_4', 'farmer_price_pct_change_1',
        'year', 'week_sin', 'week_cos',
        'usd_exchange_rate', # NO DIESEL
        'no_of_holidays',
        'reg_rain', 'reg_temp',
        'retail_price_lag_1', 'retail_price_lag_2',
        'retail_price_lag_3', 'retail_price_lag_4', 'retail_price_lag_8',
        'reg_rain_lag_1', 'reg_rain_lag_4', 'reg_rain_lag_8',
        'reg_temp_lag_1', 'reg_temp_lag_4', 'reg_temp_lag_8',
        'retail_price_roll_4',
        'retail_market_enc', 'vegetable_type_enc', 'vegetable_zone_enc', 'season_enc'
    ]

    X_train, y_train = train_df[features], train_df['retail_price']
    X_test,  y_test  = test_df[features],  test_df['retail_price']
    
    # ───────────────────────────────────────────────────
    # 3. Optuna Hyperparameter Tuning - XGBoost
    # ───────────────────────────────────────────────────
    print("\n--- Tuning XGBoost ---")
    def objective_xgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42
        }
        model = xgb.XGBRegressor(**param)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(objective_xgb, n_trials=10)
    print(f"Best XGBoost Params (MAPE: {study_xgb.best_value:.4f}):", study_xgb.best_params)

    # ───────────────────────────────────────────────────
    # 4. Optuna Hyperparameter Tuning - LightGBM
    # ───────────────────────────────────────────────────
    print("\n--- Tuning LightGBM ---")
    def objective_lgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'num_leaves': trial.suggest_int('num_leaves', 31, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.LGBMRegressor(**param)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(objective_lgb, n_trials=10)
    print(f"Best LightGBM Params (MAPE: {study_lgb.best_value:.4f}):", study_lgb.best_params)

    # ───────────────────────────────────────────────────
    # 5. Train Final Ensemble using Tuned Params
    # ───────────────────────────────────────────────────
    print("\nTraining Final Tuned Models...")
    final_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
    final_xgb.fit(X_train, y_train)
    
    lgb_params = study_lgb.best_params.copy()
    lgb_params['random_state'] = 42
    lgb_params['verbose'] = -1
    final_lgb = lgb.LGBMRegressor(**lgb_params)
    final_lgb.fit(X_train, y_train)

    print("Building Tuned Ensemble Predictions...")
    pred_xgb   = final_xgb.predict(X_test)
    pred_lgb   = final_lgb.predict(X_test)
    final_pred = (0.5 * pred_xgb) + (0.5 * pred_lgb)

    # ───────────────────────────────────────────────────
    # 6. Evaluation & Reporting
    # ───────────────────────────────────────────────────
    r2        = r2_score(y_test, final_pred)
    mape      = mean_absolute_percentage_error(y_test, final_pred)
    mape_xgb  = mean_absolute_percentage_error(y_test, pred_xgb)
    mape_lgb  = mean_absolute_percentage_error(y_test, pred_lgb)

    dataset_name = os.path.basename(data_path)
    report = f"""XGBoost + LightGBM Ensemble - OPTUNA TUNED (No Diesel)
===========================================================
Model built from: {dataset_name} 
Target  : retail_price
Optuna Trials: 10 per algorithm

Tuned Hyperparameters
--------------------
  XGBoost  : {study_xgb.best_params}
  LightGBM : {study_lgb.best_params}

Overall Ensemble Metrics
------------------------
  R2  Score              : {r2:.4f}
  Accuracy (1 - MAPE)    : {(1 - mape)*100:.2f}%
  MAPE                   : {mape*100:.2f}%

Individual Accuracies
---------------------
  XGBoost Accuracy       : {(1 - mape_xgb)*100:.2f}%
  LightGBM Accuracy      : {(1 - mape_lgb)*100:.2f}%
"""
    print(report)

    report_name = os.path.join(report_dir, 'xgb_lgbm_ensemble_no_diesel_optuna_performance.txt')
    with open(report_name, 'w', encoding='utf-8') as f:
        f.write(report)

    bundle = {
        'xgb'            : final_xgb,
        'lgb'            : final_lgb,
        'features'       : features,
        'label_encoders' : le_dict,
        'weights'        : {'xgb': 0.5, 'lgb': 0.5},
    }
    model_name = os.path.join(model_dir, 'xgb_lgbm_ensemble_no_diesel_optuna_model.joblib')
    joblib.dump(bundle, model_name)
    print(f"Artifacts saved explicitly to: {model_dir}")

if __name__ == "__main__":
    train_xgb_lgbm_no_diesel_optuna()

Loading data...


[I 2026-03-19 08:55:20,798] A new study created in memory with name: no-name-609eecf9-239a-4f41-8864-623e5f4c4a9a



--- Tuning XGBoost ---


[I 2026-03-19 08:55:44,480] Trial 0 finished with value: 0.09510118879889873 and parameters: {'n_estimators': 707, 'learning_rate': 0.010118376663336584, 'max_depth': 10, 'subsample': 0.7702887729281119, 'colsample_bytree': 0.6100326127717058}. Best is trial 0 with value: 0.09510118879889873.
[I 2026-03-19 08:55:51,350] Trial 1 finished with value: 0.09295832475047283 and parameters: {'n_estimators': 666, 'learning_rate': 0.01383291855945484, 'max_depth': 5, 'subsample': 0.6966128733126443, 'colsample_bytree': 0.8994573704669218}. Best is trial 1 with value: 0.09295832475047283.
[I 2026-03-19 08:56:01,651] Trial 2 finished with value: 0.09351360166088143 and parameters: {'n_estimators': 489, 'learning_rate': 0.017349754275809943, 'max_depth': 8, 'subsample': 0.764424181890788, 'colsample_bytree': 0.9898517484791132}. Best is trial 1 with value: 0.09295832475047283.
[I 2026-03-19 08:56:08,410] Trial 3 finished with value: 0.09937889273453376 and parameters: {'n_estimators': 258, 'learni

Best XGBoost Params (MAPE: 0.0930): {'n_estimators': 666, 'learning_rate': 0.01383291855945484, 'max_depth': 5, 'subsample': 0.6966128733126443, 'colsample_bytree': 0.8994573704669218}

--- Tuning LightGBM ---


[I 2026-03-19 08:56:55,514] Trial 0 finished with value: 0.09325217981272249 and parameters: {'n_estimators': 427, 'learning_rate': 0.02099814223972636, 'max_depth': 5, 'num_leaves': 33, 'subsample': 0.8968624897540733, 'colsample_bytree': 0.6851644456693593}. Best is trial 0 with value: 0.09325217981272249.
[I 2026-03-19 08:56:58,180] Trial 1 finished with value: 0.09237504560546637 and parameters: {'n_estimators': 521, 'learning_rate': 0.02958556649069663, 'max_depth': 7, 'num_leaves': 40, 'subsample': 0.8362667525230384, 'colsample_bytree': 0.7125330851470852}. Best is trial 1 with value: 0.09237504560546637.
[I 2026-03-19 08:57:00,889] Trial 2 finished with value: 0.09581954564050792 and parameters: {'n_estimators': 204, 'learning_rate': 0.01812657540399642, 'max_depth': 9, 'num_leaves': 56, 'subsample': 0.6496069733602607, 'colsample_bytree': 0.9275812579431255}. Best is trial 1 with value: 0.09237504560546637.
[I 2026-03-19 08:57:04,297] Trial 3 finished with value: 0.09324915811

Best LightGBM Params (MAPE: 0.0924): {'n_estimators': 521, 'learning_rate': 0.02958556649069663, 'max_depth': 7, 'num_leaves': 40, 'subsample': 0.8362667525230384, 'colsample_bytree': 0.7125330851470852}

Training Final Tuned Models...
Building Tuned Ensemble Predictions...
XGBoost + LightGBM Ensemble - OPTUNA TUNED (No Diesel)
Model built from: Final_Combined_data.csv 
Target  : retail_price
Optuna Trials: 10 per algorithm

Tuned Hyperparameters
--------------------
  XGBoost  : {'n_estimators': 666, 'learning_rate': 0.01383291855945484, 'max_depth': 5, 'subsample': 0.6966128733126443, 'colsample_bytree': 0.8994573704669218}
  LightGBM : {'n_estimators': 521, 'learning_rate': 0.02958556649069663, 'max_depth': 7, 'num_leaves': 40, 'subsample': 0.8362667525230384, 'colsample_bytree': 0.7125330851470852}

Overall Ensemble Metrics
------------------------
  R2  Score              : 0.9308
  Accuracy (1 - MAPE)    : 90.79%
  MAPE                   : 9.21%

Individual Accuracies
-----------